# **Q2D Optimized - Final ROBUST04 Submission**

## Strategy
- **Baseline Q2D achieved: 0.24 MAP** (vs 0.0546 with RM3-only)
- This notebook optimizes Q2D parameters and applies to all 249 queries
- Combines best practices: Gemini 3 Flash + BM25 fusion + parameter tuning

## Key Parameters to Optimize
1. **NUM_TERMS**: How many expansion terms (currently 15)
2. **NUM_SNIPPETS**: How many diverse snippets (currently 10)
3. **LAMBDA**: Fusion weight between BM25 and snippet scores (currently 0.35)
4. **BM25 k1, b**: From RM3 optimization we found k1=0.9, b=0.4 works well

## Cell 1: Setup (Java 21 + Packages)

In [ ]:
# 1. INSTALL JAVA 21
!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify
!java -version

# 2. INSTALL PACKAGES
!pip install --upgrade pip -q
!pip install pyserini trectools tabulate google-generativeai tqdm pandas -q

print("\n✅ Environment ready")

## Cell 2: Load Data & Initialize Systems

In [ ]:
from google.colab import drive, userdata
import pandas as pd
import json
import time
from tqdm import tqdm
from google.api_core import exceptions
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from tabulate import tabulate

# Mount Drive
drive.mount('/content/drive')

# Paths
BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')
MASTER_DATA_PATH = os.path.join(BASE_PATH, 'q2d_optimized_master.json')

# Load queries
queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]  # First 50 for training
ALL_QIDS = sorted(queries_df['qid'].tolist(), key=int)  # All 249 for final submission
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

print(f"✅ Loaded {len(TRAIN_QIDS)} training queries")
print(f"✅ Loaded {len(ALL_QIDS)} total queries for submission")

# Initialize Gemini
API_KEY = userdata.get('GOOGLE_API_KEY')
import google.generativeai as genai
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-3-flash-preview')

# Initialize Searcher with OPTIMIZED BM25 parameters from RM3 grid search
searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=0.9, b=0.4)  # From RM3 optimization

# Load qrels
qrels = TrecQrel(QRELS_PATH)

print("✅ Systems initialized with optimized BM25 parameters (k1=0.9, b=0.4)")

## Cell 3: Q2D Functions (Enhanced)

In [ ]:
def get_q2d_expansion(query_text, num_terms=15):
    """Generate expansion terms using Gemini"""
    prompt = f"""Identify {num_terms} most relevant specific entities, synonyms, related concepts, 
    and sub-topics for this 1990s news query: '{query_text}'. 
    Focus on: proper nouns, locations, organizations, events, technical terms.
    Comma-separated list only."""
    response = model.generate_content(prompt).text
    return [t.strip() for t in response.split(",") if t.strip()][:num_terms]

def get_q2d_snippets(query_text, terms, num_snips=10):
    """Generate diverse news snippets using Gemini"""
    prompt = f"""Query: {query_text}
    Keywords: {', '.join(terms)}
    
    Task: Write {num_snips} diverse 10-15 word news fragments from the late 1990s.
    Requirements:
    - Each fragment must use different combinations of the keywords
    - Cover different aspects/angles of the query topic
    - Use realistic 1990s news style and terminology
    - One fragment per line
    - No numbering or bullets
    """
    response = model.generate_content(prompt).text
    snippets = [s.strip() for s in response.split('\n') if s.strip()]
    # Clean up any numbering
    cleaned = []
    for s in snippets:
        # Remove leading numbers/bullets
        s = s.lstrip('0123456789.-*) ')
        if s:
            cleaned.append(s)
    return cleaned[:num_snips]

def load_master():
    """Load existing master data if available"""
    if os.path.exists(MASTER_DATA_PATH):
        with open(MASTER_DATA_PATH, 'r') as f:
            return json.load(f)
    return {}

def save_master(data):
    """Save master data to Drive"""
    with open(MASTER_DATA_PATH, 'w') as f:
        json.dump(data, f)

print("✅ Q2D functions ready")

## Cell 4: Parameter Optimization (Run on 50 Training Queries)

First, let's find the best LAMBDA value using your existing Q2D data if available, or generate new data with different parameters.

In [ ]:
# Check if we have existing Q2D data
existing_data_files = [
    os.path.join(BASE_PATH, 'q2d_part1.json'),
    os.path.join(BASE_PATH, 'q2d_rerank_master.json'),
    MASTER_DATA_PATH
]

existing_data = None
for path in existing_data_files:
    if os.path.exists(path):
        with open(path, 'r') as f:
            existing_data = json.load(f)
        print(f"✅ Found existing Q2D data: {path}")
        print(f"   Contains {len(existing_data)} queries")
        break

if existing_data and len(existing_data) >= 10:
    print(f"\n✅ Using existing data for parameter optimization")
    master_data = existing_data
else:
    print(f"\n⚠️ No sufficient existing data found.")
    print(f"   You'll need to run the data gathering loop first (see Cell 5)")
    master_data = {}

## Cell 5: LAMBDA Optimization (If You Have Data)

Test different LAMBDA values to find the best fusion weight.

In [ ]:
def evaluate_lambda(data, lambd):
    """Evaluate Q2D with specific lambda value"""
    all_rows = []
    for qid, info in data.items():
        base = info['base_bm25_scores']
        snip_info = info['snippet_rerank_scores']

        # Calculate Average Snippet Score
        avg_snip_scores = {}
        for snip_text, scores_dict in snip_info.items():
            for docid, score in scores_dict.items():
                avg_snip_scores[docid] = avg_snip_scores.get(docid, 0) + score

        num_snips = len(snip_info)
        if num_snips > 0:
            for d in avg_snip_scores:
                avg_snip_scores[d] /= num_snips

        # Linear Interpolation: (1-L)*Base + (L)*SnippetAvg
        fused = []
        for docid, b_score in base.items():
            s_score = avg_snip_scores.get(docid, 0)
            final = ((1 - lambd) * b_score) + (lambd * s_score)
            fused.append((docid, final))

        # Sort and Format for TrecRun
        fused = sorted(fused, key=lambda x: x[1], reverse=True)[:1000]
        for rank, (docid, score) in enumerate(fused, start=1):
            all_rows.append([qid, "Q0", docid, rank, score, f"Q2D_L{lambd}"])

    run_file = f'temp_q2d_lambda_{lambd}.txt'
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    te = TrecEval(TrecRun(run_file), qrels)
    return te.get_map(), te.get_precision(5), te.get_precision(10)

if master_data and len(master_data) >= 10:
    print("\n" + "="*80)
    print("LAMBDA OPTIMIZATION")
    print("="*80)
    
    lambda_results = []
    
    # Test different lambda values
    test_lambdas = [0.1, 0.2, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8]
    
    for lambd in test_lambdas:
        map_score, p5, p10 = evaluate_lambda(master_data, lambd)
        lambda_results.append([f"{lambd:.2f}", f"{map_score:.4f}", f"{p5:.4f}", f"{p10:.4f}"])
        print(f"Lambda={lambd:.2f}: MAP={map_score:.4f}")
    
    print("\n" + "="*80)
    print("LAMBDA RESULTS")
    print("="*80)
    print(tabulate(lambda_results, headers=["Lambda", "MAP", "P@5", "P@10"], tablefmt="fancy_grid"))
    
    # Find best lambda
    best_idx = max(range(len(lambda_results)), key=lambda i: float(lambda_results[i][1]))
    best_lambda = test_lambdas[best_idx]
    best_map = float(lambda_results[best_idx][1])
    
    print(f"\n🏆 BEST LAMBDA: {best_lambda:.2f}")
    print(f"   MAP: {best_map:.4f}")
    print("="*80)
else:
    print("\n⚠️ Skipping optimization - need to gather Q2D data first")
    best_lambda = 0.35  # Default

## Cell 6: Data Gathering for ALL 249 Queries

**IMPORTANT**: This cell generates Q2D data for all 249 queries.
- Uses optimized parameters
- Saves checkpoints to Drive
- Can be resumed if interrupted
- **WARNING**: This will take significant time and API calls!

In [ ]:
# CONFIGURATION FOR FINAL SUBMISSION
USE_ALL_QUERIES = True  # Set to True for final submission, False for testing
TARGET_QIDS = ALL_QIDS if USE_ALL_QUERIES else TRAIN_QIDS[:10]

# Optimized parameters
FINAL_NUM_TERMS = 15
FINAL_NUM_SNIPPETS = 10

print(f"\n🚀 Starting Q2D data gathering for {len(TARGET_QIDS)} queries...")
print(f"   NUM_TERMS: {FINAL_NUM_TERMS}")
print(f"   NUM_SNIPPETS: {FINAL_NUM_SNIPPETS}")
print(f"   BM25: k1=0.9, b=0.4")

# Load existing master data
final_master = load_master()
print(f"\n📊 Current progress: {len(final_master)}/{len(TARGET_QIDS)} queries")

# Process queries
for qid in tqdm(TARGET_QIDS, desc="Q2D Data Gathering"):
    if qid in final_master:
        continue  # Skip if already processed
    
    txt = QUERIES_DICT[qid]
    
    try:
        # 1. Expand and Generate Snippets
        terms = get_q2d_expansion(txt, num_terms=FINAL_NUM_TERMS)
        snippets = get_q2d_snippets(txt, terms, num_snips=FINAL_NUM_SNIPPETS)
        
        # 2. Get Baseline Top 1000
        base_hits = searcher.search(txt, k=1000)
        base_scores = {h.docid: float(h.score) for h in base_hits}
        top_1000_ids = set(base_scores.keys())
        
        # 3. Score Snippets (Intersecting with Top 1000)
        snippet_raw_scores = {}
        for snip in snippets:
            snip_hits = searcher.search(snip, k=1000)
            # Only save scores for docs in original BM25 top 1000
            snippet_raw_scores[snip] = {h.docid: float(h.score) for h in snip_hits if h.docid in top_1000_ids}
        
        # 4. Save
        final_master[qid] = {
            'query_text': txt,
            'expanded_terms': terms,
            'snippets': snippets,
            'base_bm25_scores': base_scores,
            'snippet_rerank_scores': snippet_raw_scores
        }
        
        # Checkpoint every query
        save_master(final_master)
        
        # Rate limiting
        time.sleep(2)
        
    except exceptions.TooManyRequests:
        print(f"\n⏳ Rate limit hit at QID {qid}. Progress saved.")
        print(f"   Completed: {len(final_master)}/{len(TARGET_QIDS)}")
        print(f"   Run this cell again in a few minutes to resume.")
        break
    except Exception as e:
        print(f"\n❌ Error on QID {qid}: {e}")
        continue

print(f"\n✅ Data gathering complete!")
print(f"   Total queries processed: {len(final_master)}/{len(TARGET_QIDS)}")
print(f"   Saved to: {MASTER_DATA_PATH}")

## Cell 7: Generate Final Submission File

Apply best LAMBDA to all queries and generate TREC-format run file.

In [ ]:
# Use best lambda from optimization, or default
try:
    FINAL_LAMBDA = best_lambda
except:
    FINAL_LAMBDA = 0.35  # Default if optimization wasn't run

print(f"\n🎯 Generating final submission with LAMBDA={FINAL_LAMBDA}")

# Load final master data
final_master = load_master()

if not final_master:
    print("\n❌ ERROR: No Q2D data found. Run Cell 6 first!")
else:
    print(f"\n📊 Processing {len(final_master)} queries...")
    
    all_rows = []
    
    for qid, info in tqdm(final_master.items(), desc="Generating final run"):
        base = info['base_bm25_scores']
        snip_info = info['snippet_rerank_scores']
        
        # Calculate Average Snippet Score
        avg_snip_scores = {}
        for snip_text, scores_dict in snip_info.items():
            for docid, score in scores_dict.items():
                avg_snip_scores[docid] = avg_snip_scores.get(docid, 0) + score
        
        num_snips = len(snip_info)
        if num_snips > 0:
            for d in avg_snip_scores:
                avg_snip_scores[d] /= num_snips
        
        # Linear Interpolation
        fused = []
        for docid, b_score in base.items():
            s_score = avg_snip_scores.get(docid, 0)
            final = ((1 - FINAL_LAMBDA) * b_score) + (FINAL_LAMBDA * s_score)
            fused.append((docid, final))
        
        # Sort and format
        fused = sorted(fused, key=lambda x: x[1], reverse=True)[:1000]
        for rank, (docid, score) in enumerate(fused, start=1):
            all_rows.append([qid, "Q0", docid, rank, score, "Q2D_optimized"])
    
    # Save final submission file
    output_file = os.path.join(BASE_PATH, 'FINAL_Q2D_SUBMISSION.txt')
    pd.DataFrame(all_rows).to_csv(output_file, sep=' ', header=False, index=False)
    
    print(f"\n" + "="*80)
    print("✅ FINAL SUBMISSION GENERATED")
    print("="*80)
    print(f"File: {output_file}")
    print(f"Queries: {len(final_master)}")
    print(f"Total lines: {len(all_rows):,}")
    print(f"Lambda: {FINAL_LAMBDA}")
    print(f"BM25 parameters: k1=0.9, b=0.4")
    print("="*80)
    
    # Validation
    queries_in_file = set([row[0] for row in all_rows])
    print(f"\n📋 Validation:")
    print(f"   Unique queries in file: {len(queries_in_file)}")
    print(f"   Expected: {len(final_master)}")
    print(f"   Docs per query: {len(all_rows) // len(queries_in_file)} avg")
    
    # If we have training qrels, evaluate on training set
    if len(final_master) >= 50:
        print(f"\n📊 Evaluating on training set (first 50 queries)...")
        te = TrecEval(TrecRun(output_file), qrels)
        print(f"\n   Training Set Performance:")
        print(f"   MAP: {te.get_map():.4f}")
        print(f"   P@5: {te.get_precision(5):.4f}")
        print(f"   P@10: {te.get_precision(10):.4f}")

## Cell 8: Quick Validation Checks

In [ ]:
# Validate submission file format
output_file = os.path.join(BASE_PATH, 'FINAL_Q2D_SUBMISSION.txt')

if os.path.exists(output_file):
    df = pd.read_csv(output_file, sep=' ', header=None, names=['qid', 'q0', 'docid', 'rank', 'score', 'run'])
    
    print("\n" + "="*80)
    print("FILE VALIDATION")
    print("="*80)
    
    checks = []
    
    # Check 1: Number of queries
    unique_queries = df['qid'].nunique()
    checks.append(["Unique queries", unique_queries, "249 expected" if unique_queries == 249 else "⚠️ Check count"])
    
    # Check 2: Docs per query
    docs_per_query = df.groupby('qid').size()
    all_1000 = all(docs_per_query == 1000)
    checks.append(["Docs per query", f"{docs_per_query.min()}-{docs_per_query.max()}", "✅" if all_1000 else "⚠️ Should be 1000"])
    
    # Check 3: Ranking order
    sample_qid = df['qid'].iloc[0]
    sample_scores = df[df['qid'] == sample_qid]['score'].values
    is_sorted = all(sample_scores[i] >= sample_scores[i+1] for i in range(len(sample_scores)-1))
    checks.append(["Scores descending", "Yes" if is_sorted else "No", "✅" if is_sorted else "❌"])
    
    # Check 4: Format
    all_q0 = all(df['q0'] == 'Q0')
    checks.append(["Q0 column correct", "Yes" if all_q0 else "No", "✅" if all_q0 else "❌"])
    
    # Check 5: No duplicates per query
    has_dups = any(df.groupby('qid')['docid'].apply(lambda x: x.duplicated().any()))
    checks.append(["No duplicate docs", "No dups" if not has_dups else "Has dups!", "✅" if not has_dups else "❌"])
    
    print(tabulate(checks, headers=["Check", "Result", "Status"], tablefmt="fancy_grid"))
    
    print(f"\n📊 File Statistics:")
    print(f"   Total lines: {len(df):,}")
    print(f"   File size: {os.path.getsize(output_file):,} bytes")
    print(f"   Sample lines:")
    print(df.head(3).to_string(index=False))
    
else:
    print(f"\n❌ File not found: {output_file}")
    print("   Run Cell 7 first!")

## Summary

### Workflow:
1. ✅ Cell 1: Setup environment
2. ✅ Cell 2: Load data and initialize systems
3. ✅ Cell 3: Define Q2D functions
4. ⚙️ Cell 4: Check for existing data
5. ⚙️ Cell 5: Optimize LAMBDA (if data available)
6. 🚀 Cell 6: **Generate Q2D data for all 249 queries** (MAIN TASK)
7. 📝 Cell 7: Generate final submission file
8. ✅ Cell 8: Validate submission

### Key Parameters:
- **BM25**: k1=0.9, b=0.4 (from RM3 optimization)
- **Q2D**: 15 terms, 10 snippets
- **Fusion**: LAMBDA ≈ 0.35 (or optimized value)
- **Model**: Gemini 3 Flash

### Expected Performance:
- Based on your results: **≈ 0.24 MAP** on test set
- This is **4.4x better** than RM3-only approach (0.0546 MAP)